# **Audio-Visual Learning**

Audio and vision are naturally **synchronized**: a slamming door makes a sound at the moment it shuts; lips move while speech is produced. Audio-visual (AV) learning exploits this temporal and semantic correspondence so that one modality supervises or complements the other — often with little or no manual labeling, because the *co-occurrence itself* is the training signal.

## **1. Encoding the two streams**

- **Audio** is rarely fed as a raw waveform. It is converted to a **spectrogram** (usually a **mel-spectrogram**) — a 2D time×frequency image of energy — then encoded with a **CNN** or an audio **transformer** (e.g. AST, the Audio Spectrogram Transformer, or wav2vec-style encoders on raw audio).
- **Vision** is encoded per-frame (CNN/ViT) or spatiotemporally (3D-CNN, video transformer) to capture motion.

Because the spectrogram is image-like, both streams can share similar backbone designs, which simplifies fusion.

## **2. Audio-visual correspondence (AVC)**

The foundational self-supervised task: given an audio clip and a video frame/clip, decide whether they come from the **same moment of the same video** (positive) or are mismatched (negative). Training a network to solve AVC — with a contrastive or binary-matching loss — forces both encoders to learn semantically meaningful, *aligned* features without any labels. This is the audio-visual analogue of CLIP's image-text contrastive objective.

A closely related task is **temporal synchronization**: predict the time offset between audio and video, which sharpens fine-grained alignment.

## **3. Core tasks**

- **Lip-sync / synchronization** — detect or enforce alignment between lip motion and speech audio (e.g. SyncNet); used for dubbing, deepfake detection, and active-speaker detection.
- **Sound source localization** — "which pixels make this sound?" The model produces a heatmap over the frame highlighting the object responsible for the audio, learned from AV correspondence alone.
- **Audio-visual speech recognition (AVSR)** — combine the audio stream with **visual lip reading**; the video is most valuable when audio is noisy, improving robustness over audio-only ASR.
- **Sound separation / "the sound of pixels"** — use the video to separate or extract the audio of a specific visible source.

## **4. Sketch: mel-spectrogram + simple AVC setup**

```python
import torch
import torchaudio
import torch.nn.functional as F

# 1) audio -> mel-spectrogram (the "image" the audio encoder sees)
mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000, n_fft=400, hop_length=160, n_mels=64)
waveform, sr = torchaudio.load("clip.wav")   # (channels, samples)
spec = mel(waveform)                         # (channels, n_mels, time)
log_spec = torch.log(spec + 1e-6)            # log-compression is standard

# 2) audio-visual correspondence: align per-clip embeddings contrastively
# audio_enc: CNN/transformer over log_spec ; video_enc: CNN/3D-CNN over frames
def avc_loss(audio_emb, video_emb, temperature=0.07):
    a = F.normalize(audio_emb, dim=-1)
    v = F.normalize(video_emb, dim=-1)
    logits = a @ v.t() / temperature         # (N, N) matched pairs on diagonal
    labels = torch.arange(len(logits))
    return 0.5 * (F.cross_entropy(logits, labels) +
                  F.cross_entropy(logits.t(), labels))
```

**References**
- Arandjelović & Zisserman, *Objects that Sound* / *Look, Listen and Learn*, 2017–2018.
- Chung & Zisserman, *Out of Time: Automated Lip Sync in the Wild* (SyncNet), 2016.
- Gong et al., *AST: Audio Spectrogram Transformer*, 2021.
- Zhao et al., *The Sound of Pixels*, 2018.